# Scam Detection Using Machine Learning — Part 2

## TF-IDF Vectorizer | NLP with Scikit-Learn

This notebook focuses on TF-IDF Vectorization for NLP-based scam-message classification.

**0 = Legitimate, 1 = Scam**

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 1. Create Dataset

In [3]:
data = {
    "message": [
        "Your bank statement is ready. Check the official banking app.",
        "Congratulations! You won a cash prize. Click the link to claim now.",
        "Your order has been shipped and will arrive tomorrow.",
        "Urgent! Your account will be blocked. Send your OTP immediately.",
        "Your appointment has been confirmed for tomorrow.",
        "You won a lottery. Pay a small fee to receive your reward.",
        "Your monthly bill is available in the official portal.",
        "Send your password and OTP to verify your account.",
        "Your meeting is scheduled for Monday.",
        "Claim your reward by sending your bank details now.",
        "Your payment was received successfully.",
        "Limited time offer! Click here and provide your card details."
    ],
    "label": [0,1,0,1,0,1,0,1,0,1,0,1]
}
df = pd.DataFrame(data)
df

,message,label
0,Your bank statement is ready. Check the offici...,0
1,Congratulations! You won a cash prize. Click t...,1
2,Your order has been shipped and will arrive to...,0
3,Urgent! Your account will be blocked. Send you...,1
4,Your appointment has been confirmed for tomorrow.,0
5,You won a lottery. Pay a small fee to receive ...,1
6,Your monthly bill is available in the official...,0
7,Send your password and OTP to verify your acco...,1
8,Your meeting is scheduled for Monday.,0
9,Claim your reward by sending your bank details...,1


## 2. Train-Test Split

In [4]:
X = df["message"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 9
Testing samples: 3


## 3. TF-IDF Vectorizer

TF-IDF converts text into numerical features. Fit it only on training data and use transform on test/new data.

In [5]:
tfidf = TfidfVectorizer(lowercase=True, stop_words="english")

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

print("Training matrix:", X_train_tfidf.shape)
print("Testing matrix:", X_test_tfidf.shape)

Training matrix: (9, 43)
Testing matrix: (3, 43)


## 4. Train Logistic Regression

In [6]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)
print("Model training completed.")

Model training completed.


## 5. Prediction and Evaluation

In [7]:
y_pred = model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(
    y_test, y_pred,
    target_names=["Legitimate", "Scam"],
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.3333333333333333

Classification Report:
              precision    recall  f1-score   support

  Legitimate       0.00      0.00      0.00         2
        Scam       0.33      1.00      0.50         1

    accuracy                           0.33         3
   macro avg       0.17      0.50      0.25         3
weighted avg       0.11      0.33      0.17         3

Confusion Matrix:
[[0 2]
 [0 1]]


## 6. Predict New Messages

In [8]:
new_messages = [
    "Urgent! You won a prize. Send your bank details to claim it.",
    "Your order has been delivered successfully."
]

new_tfidf = tfidf.transform(new_messages)
predictions = model.predict(new_tfidf)

for msg, pred in zip(new_messages, predictions):
    print(("SCAM" if pred == 1 else "LEGITIMATE") + ": " + msg)

SCAM: Urgent! You won a prize. Send your bank details to claim it.
LEGITIMATE: Your order has been delivered successfully.


## 7. Inspect TF-IDF Features

In [9]:
features = tfidf.get_feature_names_out()
coefficients = model.coef_[0]

feature_importance = pd.DataFrame({
    "word": features,
    "coefficient": coefficients
}).sort_values("coefficient", ascending=False)

feature_importance.head(10)

,word,coefficient
0,account,0.262853
33,send,0.262853
23,otp,0.262853
11,click,0.231762
42,won,0.231762
41,verify,0.164013
24,password,0.164013
16,immediately,0.147197
40,urgent,0.147197
6,blocked,0.147197


## 8. Scikit-Learn Pipeline

In [10]:
from sklearn.pipeline import Pipeline

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, stop_words="english")),
    ("model", LogisticRegression(max_iter=1000))
])

pipeline.fit(X_train, y_train)

prediction = pipeline.predict([
    "Click this link immediately to claim your lottery reward."
])

print("Prediction:", "SCAM" if prediction[0] == 1 else "LEGITIMATE")

Prediction: SCAM


## Conclusion

**Text → TF-IDF → Logistic Regression → Prediction → Evaluation**

TF-IDF is a useful baseline for converting text into numerical features. A real scam-detection system needs a large, representative and properly labelled dataset.